# 🏥 Healthcare QA Chatbot - Knowledge Base Builder

Downloads medical datasets and builds the ChromaDB knowledge base.

**Run all cells in order.**

In [ ]:
# Step 1: Install dependencies (ignore dependency warnings - they're harmless)
import os
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["CHROMA_TELEMETRY"] = "False"

!pip install -q --upgrade chromadb==0.5.23 sentence-transformers datasets pandas pyarrow tqdm 2>/dev/null
print("✓ Dependencies installed")

In [ ]:
# Step 2: Import libraries
import os
os.environ["ANONYMIZED_TELEMETRY"] = "False"

import gc
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from datasets import load_dataset
import pandas as pd
from tqdm.notebook import tqdm
import numpy as np

DATA_DIR = Path("/content/data/raw")
KB_DIR = Path("/content/knowledge_base")
DATA_DIR.mkdir(parents=True, exist_ok=True)
KB_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Libraries imported")

In [ ]:
# Step 3: Download all medical datasets
print("Downloading medical datasets...\n")
datasets_info = []

def download_and_save(name, hf_path, config, output_path, subset_size=None):
    """Download dataset from HuggingFace and save as parquet."""
    try:
        if config:
            ds = load_dataset(hf_path, config)
        else:
            ds = load_dataset(hf_path)
        
        split = "train" if "train" in ds else list(ds.keys())[0]
        data = ds[split]
        
        if subset_size and len(data) > subset_size:
            data = data.select(range(subset_size))
        
        output_path.parent.mkdir(parents=True, exist_ok=True)
        data.to_parquet(output_path)
        
        datasets_info.append((name, len(data)))
        print(f"  ✓ {name}: {len(data):,} entries")
        return True
    except Exception as e:
        print(f"  ✗ {name}: {str(e)[:50]}")
        return False

# Download all datasets
download_and_save("MedQuAD", "keivalya/MedQuad-MedicalQnADataset", None, 
                  DATA_DIR / "mediqa/medquad.parquet")

download_and_save("PubMedQA", "qiaojin/PubMedQA", "pqa_labeled", 
                  DATA_DIR / "pubmed/pubmedqa.parquet")

download_and_save("MedMCQA", "openlifescienceai/medmcqa", None, 
                  DATA_DIR / "mediqa/medmcqa.parquet")

download_and_save("HealthCareMagic", "wangrongsheng/HealthCareMagic-100k-en", None, 
                  DATA_DIR / "mediqa/healthcare_magic.parquet", subset_size=50000)

download_and_save("MedQA-USMLE", "GBaker/MedQA-USMLE-4-options", None, 
                  DATA_DIR / "medqa/medqa_usmle.parquet")

download_and_save("ChatDoctor-iCliniq", "lavita/ChatDoctor-iCliniq", None, 
                  DATA_DIR / "chatdoctor/icliniq.parquet")

download_and_save("ChatDoctor-HCM", "lavita/ChatDoctor-HealthCareMagic-100k", None, 
                  DATA_DIR / "chatdoctor/healthcaremagic.parquet", subset_size=100000)

download_and_save("Meadow-WikiDoc", "medalpaca/medical_meadow_wikidoc", None, 
                  DATA_DIR / "meadow/wikidoc.parquet")

download_and_save("Meadow-PatientInfo", "medalpaca/medical_meadow_wikidoc_patient_information", None, 
                  DATA_DIR / "meadow/wikidoc_patient.parquet")

download_and_save("Meadow-MEDIQA", "medalpaca/medical_meadow_mediqa", None, 
                  DATA_DIR / "meadow/mediqa.parquet")

download_and_save("Meadow-MedQA", "medalpaca/medical_meadow_medqa", None, 
                  DATA_DIR / "meadow/medqa.parquet")

# Summary
total = sum(c for _, c in datasets_info)
print(f"\n{'='*40}")
print(f"Downloaded: {len(datasets_info)} datasets, {total:,} total entries")
print(f"{'='*40}")

In [ ]:
# Step 4: Initialize embedding model
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print(f"✓ Model loaded (dimension: {embedder.get_sentence_embedding_dimension()})")

In [ ]:
# Step 5: Initialize ChromaDB
import chromadb
from chromadb.config import Settings

print("Initializing ChromaDB...")

# Create client with telemetry disabled
client = chromadb.PersistentClient(
    path=str(KB_DIR),
    settings=Settings(anonymized_telemetry=False)
)

# Delete existing collection if exists
try:
    client.delete_collection("medical_knowledge")
except:
    pass

# Create new collection
collection = client.create_collection(
    name="medical_knowledge",
    metadata={"hnsw:space": "cosine"}
)
print(f"✓ ChromaDB ready")

In [ ]:
# Step 6: Process datasets into chunks
def safe_get(row, *keys):
    """Safely get value from row, trying multiple keys."""
    for key in keys:
        val = row.get(key)
        if val is not None and not (isinstance(val, float) and pd.isna(val)):
            return str(val).strip()
    return ""

def chunk_text(text, size=512, overlap=50):
    """Split text into chunks."""
    words = text.split()
    if len(words) <= size:
        return [text]
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i+size])
        if chunk.strip():
            chunks.append(chunk)
        i += size - overlap
    return chunks

print("Processing datasets into chunks...\n")
all_chunks = []

# Process each dataset
dataset_configs = [
    ("mediqa/medquad.parquet", "MedQuAD", ["Question", "question"], ["Answer", "answer"]),
    ("pubmed/pubmedqa.parquet", "PubMedQA", ["question"], ["long_answer"]),
    ("mediqa/medmcqa.parquet", "MedMCQA", ["question"], ["exp"]),
    ("mediqa/healthcare_magic.parquet", "HealthCareMagic", ["input", "instruction"], ["output"]),
    ("chatdoctor/icliniq.parquet", "ChatDoctor", ["input", "instruction"], ["output"]),
    ("chatdoctor/healthcaremagic.parquet", "ChatDoctor", ["input", "instruction"], ["output"]),
    ("meadow/wikidoc.parquet", "MedicalMeadow", ["instruction", "input"], ["output"]),
    ("meadow/wikidoc_patient.parquet", "MedicalMeadow", ["instruction", "input"], ["output"]),
    ("meadow/mediqa.parquet", "MedicalMeadow", ["instruction", "input"], ["output"]),
    ("meadow/medqa.parquet", "MedicalMeadow", ["instruction", "input"], ["output"]),
]

for filepath, source, q_keys, a_keys in dataset_configs:
    path = DATA_DIR / filepath
    if not path.exists():
        continue
    
    df = pd.read_parquet(path)
    start_count = len(all_chunks)
    
    for _, row in df.iterrows():
        q = safe_get(row, *q_keys)
        a = safe_get(row, *a_keys)
        
        if not q or not a or len(q) < 10 or len(a) < 10:
            continue
        
        content = f"Question: {q}\n\nAnswer: {a}"
        for chunk in chunk_text(content):
            all_chunks.append({"content": chunk, "source": source})
    
    print(f"{source} ({path.name}): {len(df):,} → {len(all_chunks) - start_count:,} chunks")

# MedQA USMLE (special handling for options)
path = DATA_DIR / "medqa/medqa_usmle.parquet"
if path.exists():
    df = pd.read_parquet(path)
    start_count = len(all_chunks)
    
    for _, row in df.iterrows():
        q = safe_get(row, "question")
        options = row.get("options", {})
        answer_key = row.get("answer_idx", row.get("answer", ""))
        
        if isinstance(options, dict) and answer_key in options:
            a = str(options[answer_key])
        elif isinstance(options, list) and isinstance(answer_key, int) and 0 <= answer_key < len(options):
            a = str(options[answer_key])
        else:
            a = str(answer_key) if answer_key else ""
        
        if q and a and len(a) > 5:
            content = f"Question: {q}\n\nAnswer: {a}"
            for chunk in chunk_text(content):
                all_chunks.append({"content": chunk, "source": "MedQA-USMLE"})
    
    print(f"MedQA-USMLE: {len(df):,} → {len(all_chunks) - start_count:,} chunks")

print(f"\n{'='*40}")
print(f"Total chunks: {len(all_chunks):,}")
print(f"{'='*40}")
gc.collect()

In [ ]:
# Step 7: Index chunks into ChromaDB
print("Indexing into ChromaDB...")
print("(This takes 20-40 minutes)\n")

batch_size = 500
total = len(all_chunks)

for i in tqdm(range(0, total, batch_size), desc="Indexing"):
    batch = all_chunks[i:i + batch_size]
    texts = [c["content"] for c in batch]
    
    # Generate embeddings
    embeddings = embedder.encode(
        texts, 
        batch_size=32, 
        show_progress_bar=False,
        normalize_embeddings=True
    )
    
    # Add to ChromaDB
    collection.add(
        ids=[f"doc_{i+j}" for j in range(len(batch))],
        embeddings=embeddings.tolist(),
        documents=texts,
        metadatas=[{"source": c["source"]} for c in batch]
    )
    
    if (i // batch_size) % 100 == 0:
        gc.collect()

print(f"\n✓ Indexed {collection.count():,} documents")

In [ ]:
# Step 8: Test the knowledge base
print("Testing knowledge base...\n")

test_query = "What are the symptoms of diabetes?"
query_emb = embedder.encode([test_query], normalize_embeddings=True)

results = collection.query(
    query_embeddings=query_emb.tolist(),
    n_results=3
)

print(f"Query: {test_query}\n")
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"{i+1}. [{meta['source']}]")
    print(f"   {doc[:150]}...\n")

print("✓ Knowledge base working!")

In [ ]:
# Step 9: Create download
import shutil

print("Creating ZIP file...")
shutil.make_archive("/content/knowledge_base", 'zip', KB_DIR)

print(f"\n{'='*50}")
print("✓ BUILD COMPLETE!")
print(f"{'='*50}")
print(f"Documents indexed: {collection.count():,}")
print(f"\nDownload: /content/knowledge_base.zip")
print("Extract to your project's data/ folder")

In [ ]:
# Step 10: Download the file
try:
    from google.colab import files
    files.download("/content/knowledge_base.zip")
except:
    print("Download from Files panel on the left: /content/knowledge_base.zip")